# Importation des bibliothèques

In [ ]:
import pandas as pd
from utils import  beau_tableau, carte_par_candidat, cartes_plusieurs_candidats 

# Importation du dataset

In [ ]:
# Importation
df = pd.read_csv(
 'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb' 
)
# Aperçu
print(f"Dimension du dataset : {df.shape}")
print(f"Type des variables  : {df.dtypes}")

print("Aperçu du dataset  : ")
print(df.head(2))


On va spécifier les types des variables pour un travail plus fluide.

In [ ]:
df = df.convert_dtypes()
df.dtypes
df["code_commune"] = df["code_commune"].astype("string")


# 0. Explorations de la base de données

In [ ]:
# Aperçu de la base de données
beau_tableau(df.head(),
col_gras="all"
)

**Avant d'attaquer la question 1, on décide d'en savoir plus sur les communes et les départements pour mieux construire le code_commune.**

In [ ]:
print(df['code_commune'].str.len().value_counts())
print(df['code_departement'].str.len().value_counts())
# Voir  les codes départements uniques
print(df['code_departement'].unique())
#beau_tableau(df[df['code_departement'].str.len() == 11])
print(df[df['code_departement'].str.len() == 11]['code_departement'].unique())


On remarque que les codes des communes sont constitués de chiffres avec 3 chiffes au maximum. Pour les départements on a au maximum une chaine de 11 caractères correspondant à fr_etranger. 

In [ ]:
# Départements avec 3 chiffres
print(df[df['code_departement'].str.len() == 3])
#df[df['code_departement'] == "988"]

# Type de commune avec départements de 3 chiffres 
print(df[df['code_departement'].str.len() == 3]['code_commune'].str.len().value_counts())


In [ ]:
# # Départements de 3 chiffres dont les codes de commune n'ont pas trois chiffres
mask = (df['code_departement'].str.len() == 3) & (df['code_commune'].str.len() != 3)
df[mask][['code_departement', 'code_commune', 'libelle_commune']].drop_duplicates().head(20)

In [ ]:
# Départements de 3 chiffres dont les codes de commune ne commencent pas par le dernier chiffre du code département
# Le but est de savoir si on peut extraire les 2 premiers chiffres du code département dans tous les cas
mask = (df['code_departement'].str.len() == 3) & (df['code_commune'].str.len() == 3)
df_dom = df[mask].copy()

df_dom['dept_last'] = df_dom['code_departement'].str[-1:]
df_dom['commune_first'] = df_dom['code_commune'].str[0]

df_dom[df_dom['dept_last'] != df_dom['commune_first']][['code_departement', 'code_commune', 'dept_last', 'commune_first']].drop_duplicates().head(20)



- Vérification initiale des formats : `code_commune` a jusqu’à 3 chiffres, `code_departement` va de 2 à `fr_etranger` (11), donc normalisation nécessaire.
- Extraction des cas particuliers : départements à 3 chiffres (Dom-Tom, etc.) et communes qui ne suivent pas systématiquement la logique « 2 premiers chiffres du département + code commune ».
- Détection de contre-exemples (pour 3 chiffres département + 3 chiffres commune avec premier chiffre ≠ dernier chiffre du département), pointant la complexité d’une règle unique. : il y en a aucun
- Conclusion : il faut traiter des cas particuliers explicitement (ex. `fr_etranger` → `99`, Wallis-et-Futuna, etc.), puis harmoniser `code_commune` avec zfill et découpe contrôlée pour garantir des codes cohérents.

# 1. Explorations générales

## Question 1 :  Création et mise à jour des variables

Création ou mise à jours des  variables :
- `code_commune` : département + commune
- `candidat` : prénom + nom 

### Code commune

In [ ]:
#  Remplacer fr_etranger par 99
df['code_departement'] = df['code_departement'].replace('fr_etranger', '99')

# Construire le code commune
df['code_commune'] = (
    df['code_departement'].str[:2]
    + df['code_commune'].str[-3:].str.zfill(3) # pour completer des zeros avant le chiffre pour en avoir 3
)

# Correction Wallis-et-Futuna (986) : on prend les 3 chiffres du département
mask_wallis = df['code_departement'].astype(str) == '986'
df.loc[mask_wallis, 'code_commune'] = (
    '986' + df.loc[mask_wallis, 'code_commune'].str[-1:].str.zfill(2)
)

# Vérification
df['code_commune'].str.len().value_counts()

### Création de candidat

In [ ]:
df['candidat'] = df['prenom'] + ' ' + df['nom']
df.sample(7)

## Question 2 : Complétons la phrase grâce à Python

In [ ]:
# Préliminaires
print(df[df['prenom'].isna()]['nom'].unique()) # toutes les catégories de NA

candidats = df[df['prenom'].notna()]['candidat'].nunique()#on y enlève les na



*On peut alors écrire la phrase*

In [ ]:
# Phrase
f"En 2022, il y avait {candidats} candidats à l'élection présidentielle."

## Question 3

In [ ]:
# Votes exprimés uniquement (on exclut les lignes sans prénom)
df_exprimes = df[df['prenom'].notna()]

# Total des voix par candidat
votes_national = df_exprimes.groupby('candidat')['voix'].sum().reset_index()
votes_national.columns = ['Candidat', 'Nombre votes (total)']

# Total général pour calculer les pourcentages
total_exprimes = votes_national['Nombre votes (total)'].sum()

# Score en %
votes_national['Score (% votes exprimés)'] = (votes_national['Nombre votes (total)'] / total_exprimes * 100).round(2)

# Tri décroissant
votes_national = votes_national.sort_values('Nombre votes (total)', ascending=False).reset_index(drop=True)

votes_national.head()

*On obtient quelque chose de potable. On passee à la mise en forme.*

### Tableau mis en forme

In [ ]:

beau_tableau(
    votes_national,
    col_entiers=["Nombre votes (total)"],
    col_pourcentages=["Score (% votes exprimés)"],
    titre="Elections",
    sous_titre="Résultats du premier tour",
    date="10 avril 2022",
    note="Table 1. Résultats du premier tour de l'élection présidentielle 2022"
)

**Interprétation du tableau des résultats du premier tour de l'élection présidentielle 2022**
Le tableau montre les candidats classés par nombre de voix décroissant, avec le total des voix et le pourcentage des votes exprimés. Emmanuel Macron arrive en tête avec environ 9.78 millions de voix (27.85%), suivi de Marine Le Pen avec environ 8.13 millions (23.15%). Les autres candidats ont des scores inférieurs, avec Jean-Luc Mélenchon à environ 7.71 millions (21.95%).
Ce tableau représente les résultats nationaux agrégés, excluant les votes blancs ou nuls.

# Partie 2

La répartition départemental des votes est-elle la même que la répartition nationale ?

In [ ]:
# data frame des scores par départememt du nombre de votes et du score de chaque candidat
## Tout d'abord, on compte pour chaque candidat le nombre de votes par département
### construction du nombre total de voix pour chaque candidat
votes_departementaux = df_exprimes.groupby(['code_departement', 'candidat'])['voix'].sum().reset_index()
votes_departementaux.columns = ['code_departement', 'Candidat', 'Nombre votes (total)']

## On peut ainsi construire le score départemental de chaque candidat
### construction du score de chaque candidat (en %)

#### Total général des votes par département pour calculer les pourcentages
total_par_dep = votes_departementaux.groupby('code_departement')['Nombre votes (total)'].transform('sum')


### Score en % de chaque candidat dans chaque département
votes_departementaux['Score (% votes exprimés)'] = (
    (votes_departementaux['Nombre votes (total)'] / total_par_dep) * 100
).round(2)


# Tri décroissant par département
votes_departementaux = votes_departementaux.sort_values(['code_departement', 'Nombre votes (total)'], ascending=False).reset_index(drop=True)


print(votes_departementaux[votes_departementaux['code_departement'] == '11'])


Affichage du tableau des votes du département 11

In [ ]:
beau_tableau(votes_departementaux[votes_departementaux['code_departement'] == '11'],
#    col_departement='code_departement',
    col_entiers=['Nombre votes (total)'],
    col_pourcentages=['Score (% votes exprimés)'],
    titre="Résultats par Département",
    sous_titre="Détail des voix par candidat",
    date="2024",
    col_gras = "all"
)

# Question 5

On étudie pour chaque département la différence entre son score départemental et son score national

In [ ]:
# lien entre les résultats nationaux et départementaux
# 1. Préparation de la table nationale (on renomme pour éviter les confusions)
df_nat_clean = votes_national[['Candidat', 'Nombre votes (total)', 'Score (% votes exprimés)']].copy()
df_nat_clean.columns = ['Candidat', 'votes_national', 'score_national']

# 2. Fusion avec la table départementale
# On utilise 'left' pour garder tous les candidats du département
score_departements = pd.merge(
    votes_departementaux, 
    df_nat_clean, 
    on='Candidat', 
    how='left'
)

# 3. Renommage des colonnes départementales pour la clarté
score_departements = score_departements.rename(columns={
    'code_departement': 'code_departement',
    'Nombre votes (total)': 'votes_departement',
    'Score (% votes exprimés)': 'score_departement'
})

print(score_departements[score_departements.code_departement=="11"])



On affiche le tableau des scores par département et du score national pour chaque candidat.

In [ ]:
# affichage tableau départemental-national

# On définit les groupes de colonnes pour la fonction
colonnes_nombres = ['votes_departement', 'votes_national']
colonnes_scores = ['score_departement', 'score_national']

# Appel de votre fonction modèle
tableau_comparatif = beau_tableau(
    score_departements.query("code_departement == '11'"), 
    col_entiers=colonnes_nombres,
    col_pourcentages=colonnes_scores,
    titre="Analyse Comparative des Résultats",
    sous_titre="Comparaison des scores départementaux vs moyenne nationale",
    date="2024",
    note="Source : Fusion des données locales et nationales",
    col_gras = "all"
)

# Affichage
tableau_comparatif

# Question 6

Certains candidats sont surreprésentés dans certains départements

In [ ]:
# Construction de la variable de surrepresentation

score_departements['surrepresentation'] = (score_departements['score_departement'] - score_departements['score_national']) / score_departements['score_national'] * 100

print(score_departements.head())

# Question 7

La fonction ci-après permet d'afficher les départements où un candidat est la plus surreprésenté ou sous représenté

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns  # On importe 'seaborn' et on le nomme 'sns'

def plot_top_surrepresentations(data, candidat, nb_dep=10):
    # 1. Filtrage pour le candidat choisi
    df_c = data[data['Candidat'] == candidat].copy()
    
    if df_c.empty:
        print(f"⚠️ Le candidat '{candidat}' n'a pas été trouvé.")
        return

    # 2. Calcul de la valeur absolue pour trouver les écarts les plus grands
    df_c['abs_surrep'] = df_c['surrepresentation'].abs()

    # 3. Tri par valeur absolue et sélection du top N
    # On prend les N plus grands écarts (positifs ou négatifs)
    df_top = df_c.sort_values('abs_surrep', ascending=False).head(nb_dep)
    
    # Optionnel : On retrie par valeur réelle pour que le graphique soit ordonné de gauche à droite
    df_top = df_top.sort_values('surrepresentation', ascending=False)

    # 4. Création du graphique
    plt.figure(figsize=(10, 6))
    
    # Couleur bleue uniforme comme demandé
    sns.barplot(
        data=df_top, 
        x='surrepresentation', 
        y='code_departement', 
        color='blue',
        edgecolor='black',
        linewidth=0.5
    )

    # 5. Titre et mise en page
    plt.title(f"TOP {nb_dep} des plus grands écarts de représentation de {candidat}", fontsize=14, fontweight='bold')
    plt.xlabel("Surreprésentation (+) ou Sous-représentation (-) en %")
    plt.ylabel("Départements (Code)")
    
    # La ligne rouge à zéro devient cruciale ici pour séparer gauche/droite
    plt.axvline(x=0, color='red', linestyle='-', linewidth=1.5, alpha=0.8)
    
    # On ajoute une grille pour mieux lire les valeurs négatives
    plt.grid(axis='x', linestyle='--', alpha=0.3)
    
    plt.tight_layout()
    plt.show()



Exemple pour Eric Zemmour

In [ ]:
## affichage pour Eric Zemmour
plot_top_surrepresentations(data=score_departements, candidat='Éric ZEMMOUR', nb_dep=5)

## affichage pour Emmannuel Macron
##plot_top_surrepresentations(data=score_departements, candidat='Emmanuel MACRON', nb_dep=5)

# 3. Un peu de cartographie

## Données

Nous importons ici les données geojson contenant le fond de carte  des départements de France.

In [ ]:
departement_borders = carti_download(
    values=["France"],
    crs=4326,
    borders="DEPARTEMENT",
    vectorfile_format="geojson",
    simplification=50,
    filter_by="FRANCE_ENTIERE_DROM_RAPPROCHES",
    source="EXPRESS-COG-CARTO-TERRITOIRE",
    year=2022
)

departement_borders.head()

In [ ]:
score_departements = votes_departementaux.copy()

score_departements.head()

## Carte pour Marine LE PEN

Dans cette partie, nous avons construit une visualisation cartographique des scores électoraux par département en structurant le code en plusieurs fonctions qui se trouvent dans le fichier **utils.py**.

Tout d’abord, la fonction `filtrer_candidat` permet de restreindre la base `score_departements` à un candidat donné, sans modifier les données initiales. Les données ainsi filtrées sont ensuite fusionnées avec le fond de carte des départements via la fonction `merge_carte`, afin d’associer à chaque département son score électoral.

La représentation cartographique est ensuite réalisée grâce à la fonction `plot_carte`, qui génère une carte choroplèthe en paramétrant notamment les couleurs, la légende et les contours des départements pour améliorer la lisibilité. Enfin, les fonctions `carte_par_candidat` et `cartes_plusieurs_candidats` permettent respectivement d’afficher la carte pour un candidat donné ou d’automatiser la génération pour plusieurs candidats.


La carte des scores de Marine LE PEN met en évidence une forte structuration territoriale de son électorat. On observe des scores particulièrement élevés dans le nord et le nord-est de la France, ainsi que dans certains départements du sud, traduisant un ancrage important dans ces zones.

À l’inverse, les scores apparaissent plus faibles dans l’ouest du pays, ce qui suggère une moindre implantation dans ces régions. Ces contrastes illustrent des disparités géographiques marquées, pouvant être liées à des facteurs socio-économiques ou encore la structure démographique.

La carte permet ainsi de visualiser clairement les zones de force et de faiblesse du candidat, offrant une lecture synthétique et immédiate des dynamiques électorales sur le territoire.

In [ ]:
## Carte pour Marine LE PEN
carte_par_candidat(score_departements, departement_borders, "Marine LE PEN",epais_lim=1.4,long_legend= 10)

Nous présentons ci-dessous les cartes de tous les candidats en utilisant la fonction `cartes_plusieurs_candidats` présenté ci-haut.

In [ ]:
liste_candidats = score_departements["Candidat"].unique().tolist()

cartes_plusieurs_candidats(
    df=score_departements,
    df_geo=departement_borders,
    liste_candidats=liste_candidats,
    indicateur="Score (% votes exprimés)",
    epais_lim=1.4,
    long_legend= 10
)
